# 🎬 ComfyUI + Wan2.2 (Image-to-Video) — Google Colab ワンクリック起動

[Zuntan03/EasyWan22](https://github.com/Zuntan03/EasyWan22) の構成を参考に、Google Colab 上で
**ComfyUI + ComfyUI-Manager + Wan2.2 (I2V) 関連カスタムノード + モデル一式** を自動セットアップし、
Cloudflare Tunnel 経由でブラウザから ComfyUI を操作できるようにするノートブックです。

## 使い方（生徒向け）
1. メニューの **「ランタイム」→「ランタイムのタイプを変更」** で **GPU（T4など）** を選択します。
2. メニューの **「ランタイム」→「すべてのセルを実行」** をクリックします（上から順に自動実行されます）。
3. 一番下のセルの出力に表示される **`https://xxxxx.trycloudflare.com`** のURLをクリックすると、ブラウザでComfyUIが開きます。
4. ComfyUI が開いたら、右上メニューの **「Workflow」→「Browse Templates」→「Video」** から **Wan2.2 Image to Video** のテンプレートを選ぶと、ダウンロード済みのモデルが自動で選択されたワークフローが読み込まれます。

## 注意事項
- Google Colab の無料枠（T4 GPU / VRAM 16GB）での動作を想定し、既定では **fp8スケール版**（軽量・高品質）を使用します。VRAMが厳しい場合はモデルダウンロードのセルで **GGUF版** に切り替えられます。
- モデルファイルは合計 10GB前後になります。Googleドライブに保存しておくと、次回以降の起動が速くなります（既定でON）。
- Colab は無操作が続くとランタイムが切断されます。長時間放置しないようにしてください。
- 生成した動画・画像はローカルにダウンロードするか、Googleドライブ保存先（`ComfyUI/output`）から取得してください。


---
## Step 0. GPUの確認


In [ ]:
gpu_check = get_ipython().getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
if not gpu_check or 'failed' in gpu_check[0].lower() or 'no devices' in gpu_check[0].lower():
    print('\u26a0\ufe0f GPUが有効になっていません。')
    print('上部メニューの「ランタイム」→「ランタイムのタイプを変更」で GPU（T4など）を選択してから、')
    print('もう一度「ランタイム」→「すべてのセルを実行」してください。')
else:
    print('\u2705 GPU:', gpu_check[0])


---
## Step 1. 保存先の設定（Googleドライブ利用）
チェックを入れると Google ドライブにインストールされ、モデルの再ダウンロードが不要になります（推奨）。
チェックを外すと Colab のローカルディスクにインストールされます（セッション終了で消えます）。


In [ ]:
import os

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/ComfyUI_Wan22'
else:
    BASE_DIR = '/content/ComfyUI_Wan22'

os.makedirs(BASE_DIR, exist_ok=True)
COMFYUI_DIR = os.path.join(BASE_DIR, 'ComfyUI')
print('BASE_DIR   =', BASE_DIR)
print('COMFYUI_DIR =', COMFYUI_DIR)


---
## Step 2. ComfyUI 本体のインストール


In [ ]:
if not os.path.isdir(os.path.join(COMFYUI_DIR, '.git')):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git "{COMFYUI_DIR}"
else:
    print('[skip] ComfyUI は既にインストール済みです。')

%cd {COMFYUI_DIR}
!pip install -q -r requirements.txt
print('\u2705 ComfyUI 本体のインストールが完了しました。')


---
## Step 3. ComfyUI-Manager と Wan2.2 関連カスタムノードのインストール


In [ ]:
CUSTOM_NODES_DIR = os.path.join(COMFYUI_DIR, 'custom_nodes')
os.makedirs(CUSTOM_NODES_DIR, exist_ok=True)

def clone_node(url, name=None):
    name = name or url.rstrip('/').split('/')[-1].replace('.git', '')
    dest = os.path.join(CUSTOM_NODES_DIR, name)
    if os.path.isdir(os.path.join(dest, '.git')):
        print(f'[skip] {name} は既にインストール済みです。')
        return
    get_ipython().system(f'git clone --depth 1 {url} "{dest}"')
    req = os.path.join(dest, 'requirements.txt')
    if os.path.isfile(req):
        get_ipython().system(f'pip install -q -r "{req}"')

# ComfyUI管理・拡張機能マネージャー
clone_node('https://github.com/ltdrdata/ComfyUI-Manager.git')
# Wan2.2 / WanVideo 系ノード（Kijai版・高度なワークフロー用）
clone_node('https://github.com/kijai/ComfyUI-WanVideoWrapper.git')
clone_node('https://github.com/kijai/ComfyUI-KJNodes.git')
# GGUF量子化モデルを読み込むためのノード
clone_node('https://github.com/city96/ComfyUI-GGUF.git')
# フレーム列を動画ファイルに書き出すためのノード
clone_node('https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git')

print('\u2705 カスタムノードのインストールが完了しました。')


---
## Step 4. Wan2.2 (I2V) モデルのダウンロード
- **fp8（推奨）**: 画質を保ったまま軽量化された版。T4 GPU（VRAM16GB）でも動作します。ComfyUI公式テンプレートと同じモデルです。
- **GGUF**: さらに軽量化したい場合の量子化版。量子化レベル（Q3〜Q8）を選べます。数値が小さいほど軽量・低画質になります。
  - GGUF版の配布リポジトリは頻繁に変わるため、`GGUF_REPO` は既定値で失敗する場合は
    [huggingface.co で `Wan2.2-I2V-A14B-GGUF` を検索](https://huggingface.co/models?search=Wan2.2-I2V-A14B-GGUF)して
    見つかったリポジトリID（例: `ユーザー名/リポジトリ名`）に書き換えてください。

モデルファイルは Hugging Face 上のファイル名を検索して取得するため、多少のファイル名変更があっても動作するようになっています。


In [ ]:
!pip install -q -U huggingface_hub

MODEL_SOURCE = "fp8 (\u63a8\u5968\u30fb\u9ad8\u54c1\u8cea)"  #@param ["fp8 (\u63a8\u5968\u30fb\u9ad8\u54c1\u8cea)", "gguf (\u7701VRAM)"]
GGUF_QUANT = "Q4_K_M"  #@param ["Q3_K_M", "Q4_K_M", "Q5_K_M", "Q6_K", "Q8_0"]
GGUF_REPO = "QuantStack/Wan2.2-I2V-A14B-GGUF"  #@param {type:"string"}


In [ ]:
import shutil
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()

MODELS_DIR = os.path.join(COMFYUI_DIR, 'models')
DIFFUSION_DIR = os.path.join(MODELS_DIR, 'diffusion_models')
VAE_DIR = os.path.join(MODELS_DIR, 'vae')
TEXT_ENC_DIR = os.path.join(MODELS_DIR, 'text_encoders')

for d in (DIFFUSION_DIR, VAE_DIR, TEXT_ENC_DIR):
    os.makedirs(d, exist_ok=True)

def find_and_download(repo_id, must_contain, dest_dir, exts=('.safetensors', '.gguf')):
    """dest_dir 直下で条件に合うファイルを探し、無ければ HuggingFace から検索してダウンロードする"""
    existing = [f for f in os.listdir(dest_dir)
                if os.path.isfile(os.path.join(dest_dir, f)) and all(s.lower() in f.lower() for s in must_contain)]
    if existing:
        print(f'[skip] {existing[0]} は既に存在します。')
        return

    files = api.list_repo_files(repo_id)
    matches = sorted(f for f in files if f.endswith(exts) and all(s.lower() in f.lower() for s in must_contain))
    if not matches:
        raise FileNotFoundError(
            f'{repo_id} 内に条件 {must_contain} に一致するファイルが見つかりませんでした。\n'
            f'リポジトリ内のファイル例: {files[:15]}\n'
            f'（GGUF_QUANT の値を変えるか、リポジトリIDを確認してください）'
        )
    filename = matches[0]
    print(f'Downloading: {repo_id}/{filename}')
    path = hf_hub_download(repo_id=repo_id, filename=filename, local_dir=dest_dir)

    flat_path = os.path.join(dest_dir, os.path.basename(filename))
    if os.path.abspath(path) != os.path.abspath(flat_path):
        shutil.move(path, flat_path)
        nested_root = os.path.join(dest_dir, filename.split('/')[0])
        if os.path.isdir(nested_root):
            shutil.rmtree(nested_root, ignore_errors=True)
    print(f'  -> {flat_path}')

# ComfyUI公式ワークフローテンプレート（video_wan2_2_14B_i2v）で使われているものと同じリポジトリ
DIFFUSION_VAE_REPO = 'Comfy-Org/Wan_2.2_ComfyUI_Repackaged'
TEXT_ENCODER_REPO = 'Comfy-Org/Wan_2.1_ComfyUI_repackaged'

# 'i2v' を必ず条件に含める（t2v / fun_control / fun_camera など同じ high_noise/low_noise
# 命名を持つ他タスク向けファイルと誤って一致しないようにするため）
if MODEL_SOURCE.startswith('fp8'):
    find_and_download(DIFFUSION_VAE_REPO, ['diffusion_models', 'i2v', 'high_noise', 'fp8_scaled'], DIFFUSION_DIR)
    find_and_download(DIFFUSION_VAE_REPO, ['diffusion_models', 'i2v', 'low_noise', 'fp8_scaled'], DIFFUSION_DIR)
else:
    find_and_download(GGUF_REPO, ['i2v', 'high_noise', GGUF_QUANT.lower()], DIFFUSION_DIR)
    find_and_download(GGUF_REPO, ['i2v', 'low_noise', GGUF_QUANT.lower()], DIFFUSION_DIR)

# VAE / テキストエンコーダー
# 'wan_2.1' を条件に含める（同じ repo にある 5B モデル用 wan2.2_vae.safetensors と
# 誤って一致しないようにするため。14B I2V は wan_2.1_vae.safetensors を使う）
find_and_download(DIFFUSION_VAE_REPO, ['vae', 'wan_2.1'], VAE_DIR)
find_and_download(TEXT_ENCODER_REPO, ['text_encoders', 'umt5'], TEXT_ENC_DIR)

print('\n\u2705 すべてのモデルのダウンロードが完了しました。')


---
## Step 5. Cloudflare Tunnel (cloudflared) の準備


In [ ]:
CLOUDFLARED_PATH = '/content/cloudflared'

if not os.path.isfile(CLOUDFLARED_PATH):
    !wget -q -O {CLOUDFLARED_PATH} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x {CLOUDFLARED_PATH}
    print('\u2705 cloudflared をインストールしました。')
else:
    print('[skip] cloudflared は既にインストール済みです。')


---
## Step 6. ComfyUI の起動 ＋ 公開URLの発行
実行すると ComfyUI がバックグラウンドで起動し、Cloudflare Tunnel の公開URLが表示されます。
URLが表示されたら、そのリンクをクリックしてブラウザで開いてください（初回はモデル読み込みで数十秒かかることがあります）。


In [ ]:
import subprocess, sys, time, re

%cd {COMFYUI_DIR}

COMFYUI_PORT = 8188

# start_new_session=True: このセルの実行が終わったり、Colabのカーネルに
# 割り込み(SIGINT/SIGTERM)が送られたりしても、バックグラウンドプロセスが
# 巻き添えで終了しないよう、ノートブックのプロセスグループから切り離す。
comfy_log = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen(
    [sys.executable, '-u', 'main.py', '--listen', '0.0.0.0', '--port', str(COMFYUI_PORT)],
    stdout=comfy_log, stderr=subprocess.STDOUT, cwd=COMFYUI_DIR,
    start_new_session=True,
)
print(f'\u25b6\ufe0f ComfyUI をバックグラウンドで起動しました (PID: {comfy_proc.pid})')

tunnel_log_path = '/content/cloudflared.log'
tunnel_log = open(tunnel_log_path, 'w')
# stdbuf -oL でcloudflaredの標準出力を行単位フラッシュに強制する。
# ファイルにリダイレクトするとブロック単位バッファリングになり、
# URLが実際にログへ書き出されるまで時間がかかることがあるための対策。
# --protocol http2: 既定のQUIC(UDP)はColabのネットワーク環境で不安定になりやすく、
# 接続してもCloudflare側で 'Error 1033 (Tunnel error)' になることがあるため、
# より安定するHTTP/2を明示的に指定する。
tunnel_proc = subprocess.Popen(
    ['stdbuf', '-oL', '-eL', CLOUDFLARED_PATH, 'tunnel', '--url', f'http://127.0.0.1:{COMFYUI_PORT}',
     '--protocol', 'http2', '--no-autoupdate'],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f'\u25b6\ufe0f Cloudflare Tunnel をバックグラウンドで起動しました (PID: {tunnel_proc.pid})')

print('\n\u23f3 公開URLの発行を待っています...')

url = None
tunnel_crashed = False
MAX_WAIT_SEC = 360
POLL_SEC = 2
for i in range(MAX_WAIT_SEC // POLL_SEC):
    time.sleep(POLL_SEC)
    with open(tunnel_log_path) as f:
        content = f.read()
    m = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', content)
    if m:
        url = m.group(0)
        break
    # cloudflared が起動直後に落ちていないか確認する。
    # （不正なフラグ等で即終了した場合、そのまま待ち続けても永遠にURLは出ない）
    if tunnel_proc.poll() is not None:
        tunnel_crashed = True
        break
    elapsed = (i + 1) * POLL_SEC
    if elapsed % 30 == 0:
        print(f'  ...まだ待機中です（経過 {elapsed}秒）。トンネル確立まで数分かかる場合があります。')

if tunnel_crashed:
    print('\n\u26a0\ufe0f cloudflared が起動直後に終了しました。以下はログの内容です:\n')
    with open(tunnel_log_path) as f:
        print(f.read())
# URLが発行された直後はCloudflareのエッジへの接続登録がまだ完了しておらず、
# すぐ開くと 'Error 1033 (Tunnel error)' になることがあるため、
# 'Registered tunnel connection'（接続確立完了）のログが出るまで少し待つ。
# ログの文言が変わっていても無限待機しないよう、最大15秒で打ち切って先に進む。
if url:
    for _ in range(15):
        with open(tunnel_log_path) as f:
            content = f.read()
        if 'Registered tunnel connection' in content:
            break
        time.sleep(1)

if url:
    print('\n' + '=' * 64)
    print('\u2705 下のURLをクリックして ComfyUI を開いてください:')
    print()
    print(url)
    print()
    print('（Error 1033 や真っ白な画面が出た場合は、接続確立中の可能性があります。')
    print('  10〜30秒ほど待ってから再読み込みしてください）')
    print('=' * 64)
elif not tunnel_crashed:
    print('\u26a0\ufe0f Tunnel URL の取得にタイムアウトしました。下のセルで /content/cloudflared.log を確認してください。')


---
### （トラブル時）ログの確認用セル
ComfyUIが起動しない・エラーが出る場合はこのセルを実行してログを確認してください。


In [ ]:
print('--- ComfyUI ログ（末尾40行） ---')
!tail -n 40 /content/comfyui.log
print()
print('--- cloudflared ログ（末尾20行） ---')
!tail -n 20 /content/cloudflared.log


---
### （終了時）プロセス停止用セル
作業が終わったら実行してください（Colabのランタイムを切断すれば自動的に停止します）。


In [ ]:
for p, name in [(globals().get('comfy_proc'), 'ComfyUI'), (globals().get('tunnel_proc'), 'cloudflared')]:
    if p is not None:
        p.terminate()
        print(f'{name} (PID: {p.pid}) を停止しました。')
    else:
        print(f'{name} のプロセスは見つかりませんでした。')
